In [1]:
# ================================================================================
# A2A Multi-Agent Demo — Fully Optimized & Fixed (Generic)
# a2a-sdk==1.0.1 | LangChain + Groq | DB via HTTP | Chat History (JSON)
# ================================================================================

# ── STEP 0: Install ──────────────────────────────────────────────────────────────
import subprocess, sys
print("📦 Installing dependencies...", flush=True)
r = subprocess.run([sys.executable, "-m", "pip", "install",
     "a2a-sdk==1.0.1", "uvicorn", "httpx", "starlette", "psutil",
     "langchain-groq", "langchain-core", "-q"], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("pip install failed:\n" + r.stderr[-2000:])
print("✅ Dependencies installed!", flush=True)

# ── STEP 1: Imports ──────────────────────────────────────────────────────────────
import asyncio, threading, time, socket, json, sqlite3, re, os
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional, Any
from concurrent.futures import ThreadPoolExecutor

import httpx, uvicorn, psutil
from starlette.applications import Starlette
from starlette.responses import JSONResponse
from starlette.routing import Route
from starlette.requests import Request
from google.protobuf.json_format import ParseDict, MessageToDict

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool

from a2a.server.agent_execution.agent_executor import AgentExecutor
from a2a.server.agent_execution.context import RequestContext
from a2a.server.events.event_queue import EventQueue
from a2a.server.tasks.inmemory_task_store import InMemoryTaskStore
from a2a.server.request_handlers.default_request_handler_v2 import DefaultRequestHandlerV2
from a2a.server.agent_execution.simple_request_context_builder import SimpleRequestContextBuilder
from a2a.server.events.in_memory_queue_manager import InMemoryQueueManager
from a2a.client.card_resolver import A2ACardResolver, AGENT_CARD_WELL_KNOWN_PATH
from a2a.types import AgentCard, Message, SendMessageRequest, UnsupportedOperationError

print("✅ All imports OK!", flush=True)

# ── STEP 2: Config dataclass ─────────────────────────────────────────────────────
@dataclass(frozen=True)
class Config:
    GROQ_API_KEY: str = "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"
    GROQ_MODEL: str = "llama-3.1-8b-instant"
    DB_API_URL: str = "http://127.0.0.1:8998"
    DB_PATH: str = "company.db"
    CHAT_HISTORY_FILE: str = "chat_history.json"
    OPENWEATHER_API_KEY: str = "9067ae13fd35fb41e06770752227e150"
    METAL_PRICES: dict = field(default_factory=lambda: {
        "gold": 88.1553, "silver": 1.0523, "platinum": 32.169,
        "copper": 0.0098, "aluminum": 2.4500, "zinc": 3.1200,
    })
    AGENT_PORTS: dict = field(default_factory=lambda: {
        "Research Agent": 9001, "Math Agent": 9002,
        "Creative Writer Agent": 9003, "Summary Agent": 9004,
        "Tool Agent": 9005, "DB Agent": 9006, "Weather Agent": 9007,
    })
    ORCHESTRATOR_PORT: int = 9000
    DB_API_PORT: int = 8998

CFG = Config()

# ── STEP 3: A2A Message helpers ──────────────────────────────────────────────────
def new_agent_text_message(text: str) -> Message:
    return ParseDict({"role": "agent", "parts": [{"text": {"text": text}}]}, Message())

def new_user_text_message(text: str) -> Message:
    return ParseDict({"role": "user", "parts": [{"text": {"text": text}}]}, Message())

# ── STEP 4: LangChain Groq Setup ─────────────────────────────────────────────────
_llm_cache: dict = {}

def get_llm(temperature: float = 0.3) -> ChatGroq:
    if temperature not in _llm_cache:
        _llm_cache[temperature] = ChatGroq(
            api_key=CFG.GROQ_API_KEY, model=CFG.GROQ_MODEL, temperature=temperature
        )
    return _llm_cache[temperature]

def call_llm(system_prompt: str, user_query: str, temperature: float = 0.3) -> str:
    return get_llm(temperature).invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_query),
    ]).content.strip()

print(f"✅ LangChain-Groq ready — Model: {CFG.GROQ_MODEL}", flush=True)

# ── STEP 5: Chat History ─────────────────────────────────────────────────────────
class ChatHistory:
    def __init__(self, filepath: str = CFG.CHAT_HISTORY_FILE):
        self.filepath = filepath
        self.history: list[dict] = []
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self._load()

    def _load(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    self.history = json.load(f)
                print(f"📚 Loaded {len(self.history)} record(s)", flush=True)
            except Exception as e:
                print(f"⚠️ Could not load history: {e}", flush=True)
                self.history = []
        else:
            print(f"📝 New chat history file", flush=True)

    def _write(self):
        try:
            with open(self.filepath, "w", encoding="utf-8") as f:
                json.dump(self.history, f, indent=2, ensure_ascii=False)
        except Exception as e:
            print(f"⚠️ Could not save history: {e}", flush=True)

    def add(self, query: str, steps: list[dict], response: str, mode: str) -> dict:
        entry = {
            "id": len(self.history) + 1,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "session": self.session_id,
            "mode": mode,
            "query": query,
            "steps": [{"step": s["step"], "agent": s["agent"],
                       "task": s["task"], "result": s["result"]} for s in steps],
            "response": response,
        }
        self.history.append(entry)
        self._write()
        return entry

    def show_recent(self, n: int = 5):
        if not self.history:
            print("  📭 No history yet.", flush=True); return
        recent = self.history[-n:]
        print(f"\n  {'─'*62}\n  📚 Last {len(recent)} of {len(self.history)} conversations\n  {'─'*62}", flush=True)
        for e in recent:
            tag = "🔗 MULTI " if e["mode"] == "multi" else "➡️  SINGLE"
            route = " → ".join(s["agent"] for s in e.get("steps", []))
            print(f"  [{e['id']:04d}] {e['timestamp']}  {tag}  session={e['session']}", flush=True)
            print(f"         Q : {e['query'][:65]}\n         ↳ : {route}", flush=True)
        print(f"  {'─'*62}\n", flush=True)

    def show_entry(self, entry_id: int):
        matches = [e for e in self.history if e["id"] == entry_id]
        if not matches:
            print(f"  ❌ No entry #{entry_id}", flush=True); return
        e = matches[0]
        print(f"\n  ── Entry #{e['id']} ──\n  Timestamp : {e['timestamp']}\n  Session   : {e['session']}\n  Mode      : {e['mode']}\n  Query     : {e['query']}", flush=True)
        for s in e.get("steps", []):
            print(f"  Step {s['step']} : {s['agent']} → {s['task'][:55]}", flush=True)
        print(f"\n{e['response']}\n", flush=True)

    def show_stats(self):
        if not self.history:
            print("  📭 No history.", flush=True); return
        total = len(self.history)
        singles = sum(1 for e in self.history if e["mode"] == "single")
        agent_counts: dict[str, int] = {}
        for e in self.history:
            for s in e.get("steps", []):
                agent_counts[s["agent"]] = agent_counts.get(s["agent"], 0) + 1
        print(f"\n  📊 Stats\n  {'─'*40}\n  Total : {total}  |  Single : {singles}  |  Multi : {total-singles}", flush=True)
        for agent, count in sorted(agent_counts.items(), key=lambda x: -x[1]):
            print(f"    {count:>4}×  {agent}", flush=True)

    def search(self, keyword: str):
        kw = keyword.lower()
        matches = [e for e in self.history if kw in e["query"].lower() or kw in e["response"].lower()]
        if not matches:
            print(f"  🔍 No matches for '{keyword}'", flush=True); return
        print(f"\n  🔍 {len(matches)} match(es):", flush=True)
        for e in matches[-10:]:
            print(f"  [{e['id']:04d}] {e['timestamp']}  {e['query'][:60]}", flush=True)

    def export_session(self, session_id: Optional[str] = None):
        sid = session_id or self.session_id
        entries = [e for e in self.history if e["session"] == sid]
        if not entries:
            print(f"  ❌ No entries for session {sid}", flush=True); return
        fname = f"session_{sid}.json"
        with open(fname, "w", encoding="utf-8") as f:
            json.dump(entries, f, indent=2, ensure_ascii=False)
        print(f"  ✅ Exported {len(entries)} entries → {fname}", flush=True)

chat_history = ChatHistory()
print("✅ ChatHistory ready!", flush=True)

# ── STEP 6: Tool-call helpers ───────────────────────────────────────────────────
def parse_text_tool_calls(content: str) -> list[dict]:
    calls, seen = [], set()
    def add(name: str, args_str: str):
        args_str = args_str.strip()
        args = {}
        if args_str:
            try:
                args = json.loads(args_str)
            except json.JSONDecodeError:
                kv = re.findall(r'(\w+)\s*=\s*["\']([^"\']*)["\']', args_str)
                if kv:
                    args = dict(kv)
                else:
                    bare = args_str.strip('"\'')
                    if bare:
                        args = {"raw": bare}
        key = (name, json.dumps(args, sort_keys=True))
        if key not in seen:
            seen.add(key)
            calls.append({"name": name, "args": args})

    for name, body in re.findall(r'<function=(\w+)>(.*?)</function>', content, re.DOTALL):
        add(name, body)
    for name, body in re.findall(r'<function=(\w+)>(\{[^<<]+)', content, re.DOTALL):
        add(name, body.strip())
    for name, body in re.findall(r'<function=(\w+)\(([^)]*)\)>', content):
        add(name, body)
    return calls

def extract_tool_calls(response) -> list[dict]:
    if hasattr(response, "tool_calls") and response.tool_calls:
        return [{"name": tc["name"], "args": tc["args"]} for tc in response.tool_calls]
    if hasattr(response, "content") and response.content:
        parsed = parse_text_tool_calls(response.content)
        if parsed:
            return parsed
    return []

def strip_agent_prefix(text: str) -> str:
    text = text.strip()
    while True:
        cleaned = re.sub(r'^\s*\[[^\]]*\]\s*', '', text, flags=re.IGNORECASE).strip()
        if cleaned == text:
            break
        text = cleaned
    return text

# ── STEP 7: Agent metadata ───────────────────────────────────────────────────────
AGENT_EMOJI = {
    "Research Agent": "🔬", "Math Agent": "🧮", "Creative Writer Agent": "✍️",
    "Summary Agent": "📄", "Tool Agent": "🛠️", "DB Agent": "🗄️", "Weather Agent": "🌤️",
}

# ── STEP 8: DB API Microservice ─────────────────────────────────────────────────
def setup_company_db():
    conn = sqlite3.connect(CFG.DB_PATH)
    c = conn.cursor()
    c.execute("CREATE TABLE IF NOT EXISTS employees (id INTEGER, name TEXT, role TEXT, team TEXT)")
    c.execute("CREATE TABLE IF NOT EXISTS projects (id TEXT, title TEXT, assigned_to INTEGER, status TEXT)")
    c.execute("DELETE FROM employees"); c.execute("DELETE FROM projects")
    c.executemany("INSERT INTO employees VALUES (?,?,?,?)", [
        (101, "Rajendra Dayma", "Data Science Intern", "AI Agents"),
        (102, "Alice Smith", "Senior Engineer", "Backend"),
        (103, "Bob Lee", "ML Engineer", "AI Agents"),
        (104, "Carol White", "Product Manager", "Platform"),
    ])
    c.executemany("INSERT INTO projects VALUES (?,?,?,?)", [
        ("A1", "Knowledge Graph Architecture", 101, "Active"),
        ("B2", "Multi-Agent Orchestration", 102, "In Review"),
        ("C3", "Model Fine-Tuning Pipeline", 103, "Active"),
        ("D4", "Internal Developer Portal", 104, "Planning"),
    ])
    conn.commit(); conn.close()

setup_company_db()
print("🗄️ SQLite company.db seeded!", flush=True)

async def db_api_query(request: Request) -> JSONResponse:
    try:
        body = await request.json()
    except Exception:
        return JSONResponse({"error": "Invalid JSON"}, status_code=400)
    sql = body.get("sql", "").strip()
    if not sql:
        return JSONResponse({"error": "Field 'sql' required"}, status_code=400)
    if not sql.upper().startswith("SELECT"):
        return JSONResponse({"error": "Only SELECT allowed"}, status_code=400)
    sql = sql.replace("\\'", "'")
    try:
        conn = sqlite3.connect(CFG.DB_PATH)
        c = conn.cursor()
        c.execute(sql)
        rows = c.fetchall()
        cols = [d[0] for d in c.description] if c.description else []
        conn.close()
        return JSONResponse({"columns": cols, "rows": [list(r) for r in rows], "count": len(rows)})
    except Exception as e:
        return JSONResponse({"error": f"SQL Error: {e}"}, status_code=400)

async def db_api_health(_request: Request) -> JSONResponse:
    try:
        conn = sqlite3.connect(CFG.DB_PATH)
        conn.execute("SELECT 1")
        conn.close()
        return JSONResponse({"status": "ok", "db": CFG.DB_PATH})
    except Exception as e:
        return JSONResponse({"status": "error", "detail": str(e)}, status_code=500)

async def db_api_schema(_request: Request) -> JSONResponse:
    return JSONResponse({
        "tables": {
            "employees": {
                "columns": ["id INTEGER", "name TEXT", "role TEXT", "team TEXT"],
                "sample": [
                    {"id": 101, "name": "Rajendra Dayma", "role": "Data Science Intern", "team": "AI Agents"},
                    {"id": 102, "name": "Alice Smith", "role": "Senior Engineer", "team": "Backend"},
                    {"id": 103, "name": "Bob Lee", "role": "ML Engineer", "team": "AI Agents"},
                    {"id": 104, "name": "Carol White", "role": "Product Manager", "team": "Platform"},
                ],
                "valid_team": ["AI Agents", "Backend", "Platform"],
            },
            "projects": {
                "columns": ["id TEXT", "title TEXT", "assigned_to INTEGER", "status TEXT"],
                "sample": [
                    {"id": "A1", "title": "Knowledge Graph Architecture", "assigned_to": 101, "status": "Active"},
                    {"id": "B2", "title": "Multi-Agent Orchestration", "assigned_to": 102, "status": "In Review"},
                    {"id": "C3", "title": "Model Fine-Tuning Pipeline", "assigned_to": 103, "status": "Active"},
                    {"id": "D4", "title": "Internal Developer Portal", "assigned_to": 104, "status": "Planning"},
                ],
                "valid_status": ["Active", "In Review", "Planning"],
            },
        },
        "join_hint": "projects.assigned_to → employees.id  ∴  FROM projects p JOIN employees e ON e.id = p.assigned_to",
    })

db_api_app = Starlette(debug=False, routes=[
    Route("/query", db_api_query, methods=["POST"]),
    Route("/health", db_api_health, methods=["GET"]),
    Route("/schema", db_api_schema, methods=["GET"]),
])
print("✅ DB API app built", flush=True)

async def call_db_api(sql: str) -> str:
    try:
        async with httpx.AsyncClient(timeout=10) as client:
            resp = await client.post(f"{CFG.DB_API_URL}/query", json={"sql": sql})
        data = resp.json()
    except Exception as e:
        return f"DB API unreachable: {e}"
    if "error" in data:
        return f"DB API Error: {data['error']}"
    cols, rows = data.get("columns", []), data.get("rows", [])
    if not rows:
        return "Query returned 0 rows."
    return "\n".join(", ".join(f"{k}={v}" for k, v in zip(cols, row)) for row in rows)

# ── STEP 9: LangChain Tools ─────────────────────────────────────────────────────
MEMORY_DB: list[str] = []

@tool
def get_metal_price(metal_name: str) -> str:
    """Fetches the current price per gram AND per kg for a specified metal."""
    key = metal_name.lower().strip()
    if key not in CFG.METAL_PRICES:
        return f"Metal '{metal_name}' not found. Supported: {', '.join(CFG.METAL_PRICES)}"
    p = CFG.METAL_PRICES[key]
    return f"metal={key} | price_per_gram=${p:.4f} | price_per_kg=${p*1000:.4f}"

@tool
def save_data(data: str) -> str:
    """Save a text string into the in-memory store."""
    MEMORY_DB.append(data)
    return f"✅ Saved: {data}"

@tool
def get_data() -> str:
    """Retrieve all entries saved with save_data."""
    if not MEMORY_DB:
        return "Memory store is empty."
    return "Memory store:\n" + "\n".join(f"  {i+1}. {d}" for i, d in enumerate(MEMORY_DB))

@tool
def execute_sql(sql_query: str) -> str:
    """Execute a SELECT SQL query against the company database via HTTP API endpoint."""
    try:
        return asyncio.run(call_db_api(sql_query))
    except RuntimeError:
        with ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(asyncio.run, call_db_api(sql_query)).result(timeout=15)
    except Exception as e:
        return f"DB call failed: {e}"

async def _fetch_weather(city: str) -> dict:
    url = (f"http://api.openweathermap.org/data/2.5/weather"
           f"?q={city}&appid={CFG.OPENWEATHER_API_KEY}&units=metric")
    try:
        async with httpx.AsyncClient(timeout=10) as client:
            return (await client.get(url)).json()
    except Exception as e:
        return {"cod": 500, "message": str(e)}

def format_weather(data: dict) -> str:
    if data.get("cod") != 200:
        return f"Weather API error: {data.get('message', 'unknown error')}"
    main, weather, wind, clouds = data["main"], data["weather"][0], data.get("wind", {}), data.get("clouds", {})
    return (f"🌍 {data['name']}, {data['sys']['country']}\n"
            f"  Condition   : {weather['description'].capitalize()}\n"
            f"  Temperature : {main['temp']}°C (feels like {main['feels_like']}°C)\n"
            f"  Min / Max   : {main['temp_min']}°C / {main['temp_max']}°C\n"
            f"  Humidity    : {main['humidity']}%\n"
            f"  Wind        : {wind.get('speed','N/A')} m/s  direction {wind.get('deg','N/A')}°\n"
            f"  Cloud cover : {clouds.get('all','N/A')}%\n"
            f"  Visibility  : {data.get('visibility', 'N/A')} m")

@tool
def get_weather(city: str) -> str:
    """Get current weather conditions for any city using OpenWeatherMap API."""
    try:
        data = asyncio.run(_fetch_weather(city))
    except RuntimeError:
        with ThreadPoolExecutor(max_workers=1) as pool:
            data = pool.submit(asyncio.run, _fetch_weather(city)).result(timeout=15)
    return format_weather(data)

print("✅ LangChain tools ready", flush=True)

# ── STEP 10: Generic Retry Wrapper ─────────────────────────────────────────────
async def llm_invoke_with_retry(
    primary_system: str,
    user_query: str,
    tools: list,
    fallback_system: str,
    prefix_tag: str = "",
    temperature: float = 0.0,
) -> Any:
    """
    Generic retry wrapper for Groq tool-use failures.
    Tries primary_system first; falls back to fallback_system on 400/tool_use_failed.
    """
    llm = get_llm(temperature).bind_tools(tools) if tools else get_llm(temperature)
    messages = [SystemMessage(content=primary_system), HumanMessage(content=user_query)]
    try:
        return llm.invoke(messages)
    except Exception as e:
        if "400" not in str(e) and "tool_use_failed" not in str(e):
            raise
        print(f"       ⚠️ [{prefix_tag.strip('[]')}] Groq 400 — retrying...", flush=True)
        messages = [SystemMessage(content=fallback_system), HumanMessage(content=user_query)]
        return llm.invoke(messages)

# ── STEP 11: Base Agent Executors ───────────────────────────────────────────────
class BaseAgentExecutor(AgentExecutor):
    """Template-method base: subclasses only implement direct_execute."""
    SYSTEM: str = ""
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(new_agent_text_message(await self.direct_execute(context.get_user_input())))
    async def cancel(self, _context: RequestContext, _event_queue: EventQueue) -> None:
        raise UnsupportedOperationError()

class LLMAgentExecutor(BaseAgentExecutor):
    """Agents that just call an LLM with a system prompt + temperature."""
    SYSTEM: str = ""
    TEMPERATURE: float = 0.0
    PREFIX: str = ""
    async def direct_execute(self, query: str) -> str:
        out = call_llm(self.SYSTEM, query, temperature=self.TEMPERATURE)
        return f"{self.PREFIX}\n{out}" if self.PREFIX else out

class ResearchAgentExecutor(LLMAgentExecutor):
    SYSTEM = ("You are Research Agent, an expert in science, geography, history, technology. "
              "Answer directly. Start every reply with exactly '[Research Agent]'. NEVER mention other agents.")
    TEMPERATURE = 0.1
    PREFIX = "[Research Agent]"

class MathAgentExecutor(LLMAgentExecutor):
    # FIX #17: Clarify indexing convention for sequences
    SYSTEM = ("You are Math Agent. Solve arithmetic, algebra, statistics, calculus. "
              "For sequences: clearly state your indexing convention (e.g. F(1)=1, F(2)=1). "
              "Show step-by-step working. Start every reply with exactly '[Math Agent]'. "
              "NEVER mention other agents.")
    TEMPERATURE = 0.0
    PREFIX = "[Math Agent]"

class CreativeWriterAgentExecutor(LLMAgentExecutor):
    # FIX #19: Mandatory source fidelity, forbid generic templates
    SYSTEM = ("You are Creative Writer Agent. "
              "MANDATORY: When source material is provided, use its EXACT specific details "
              "(numbers, conditions, locations) in your response. "
              "FORBIDDEN: Do not ignore provided facts or substitute generic seasonal templates. "
              "Start every reply with exactly '[Creative Writer Agent]'. "
              "NEVER mention other agents.")
    TEMPERATURE = 0.7
    PREFIX = "[Creative Writer Agent]"

class SummaryAgentExecutor(LLMAgentExecutor):
    SYSTEM = ("You are Summary Agent. Create concise summaries. "
              "Start every reply with exactly '[Summary Agent]'. NEVER mention other agents.")
    TEMPERATURE = 0.2
    PREFIX = "[Summary Agent]"
# ── STEP 12: Tool-Calling Mixin (cleaned, no _retry) ────────────────────────────
class ToolCallingMixin:
    SYSTEM: str = ""
    RETRY_SYSTEM: str = ""
    TOOLS: list = []
    PREFIX_TAG: str = ""

    async def direct_execute(self, query: str) -> str:
        tool_map = {t.name: t for t in self.TOOLS}
        try:
            response = await llm_invoke_with_retry(
                self.SYSTEM, query, self.TOOLS, self.RETRY_SYSTEM, self.PREFIX_TAG
            )
        except Exception as e:
            return f"{self.PREFIX_TAG}\n⚠️ LLM call failed: {e}"
        return await self._handle_response(query, response, tool_map)

    async def _handle_response(self, query: str, response, tool_map: dict) -> str:
        tool_calls = extract_tool_calls(response)
        if not tool_calls and hasattr(response, "content"):
            tool_calls = parse_text_tool_calls(response.content)
        if tool_calls:
            return await self._run_tools(query, tool_calls, tool_map)
        content = strip_agent_prefix(response.content.strip() if hasattr(response, "content") else "")
        return f"{self.PREFIX_TAG}\n{content}" if self.PREFIX_TAG else content

    async def _run_tools(self, query: str, tool_calls: list, tool_map: dict) -> str:
        results = []
        for tc in tool_calls:
            tname, targs = tc["name"], tc["args"]
            if "raw" in targs:
                missing = [p for p in tool_map[tname].args.keys() if p not in targs]
                if missing:
                    targs = {missing[0]: targs["raw"]}
            if tname in tool_map:
                try:
                    raw = await asyncio.to_thread(tool_map[tname].invoke, targs)
                    results.append(f"Tool '{tname}' → {raw}")
                    print(f"       🛠️ '{tname}' → {str(raw)[:80]}", flush=True)
                except Exception as te:
                    results.append(f"Tool '{tname}' failed: {te}")
            else:
                results.append(f"Unknown tool: {tname}")
        final = get_llm(0.0).invoke([
            SystemMessage(content="You are a helpful assistant. Answer clearly. No [Tag] prefix."),
            HumanMessage(content=f"User asked: {query}"),
            HumanMessage(content=f"Tool output:\n{chr(10).join(results)}\n\nClear accurate reply. No prefix tag."),
        ])
        return f"{self.PREFIX_TAG}\n{strip_agent_prefix(final.content)}"

class ToolAgentExecutor(BaseAgentExecutor, ToolCallingMixin):
    SYSTEM = ("You are Tool Agent. Tools:\n"
              "  1. get_metal_price(metal_name) — per-gram & per-kg\n"
              "  2. save_data(data) — save text note\n"
              "  3. get_data() — retrieve notes\n\n"
              "QUANTITY RULE: If user asks for a weight (e.g. 10 kg, 500 g):\n"
              "  1. Call get_metal_price  2. Multiply yourself  3. Show unit & total.\n"
              "Call EXACTLY ONE tool. Do NOT add [Tool Agent] prefix.")
    RETRY_SYSTEM = "You are Tool Agent. Call exactly one tool: get_metal_price | save_data | get_data. No explanation."
    TOOLS = [get_metal_price, save_data, get_data]
    PREFIX_TAG = "[Tool Agent]"

class WeatherAgentExecutor(BaseAgentExecutor, ToolCallingMixin):
    SYSTEM = ("You are Weather Agent. Tool: get_weather(city: str).\n"
              "CITY RULE: 'city' must be a specific city. If user mentions a country, use its capital.\n"
              "Examples: Italy→Rome | France→Paris | India→New Delhi | USA→New York | UK→London.\n"
              "After tool returns, write a friendly report. Do NOT add [Weather Agent] prefix.")
    RETRY_SYSTEM = "You are Weather Agent. Call get_weather(city: str). If country given, use capital. Call tool now. No explanation."
    TOOLS = [get_weather]
    PREFIX_TAG = "[Weather Agent]"

    async def _run_tools(self, query: str, tool_calls: list, tool_map: dict) -> str:
        results = []
        for tc in tool_calls:
            tname, targs = tc["name"], tc["args"]
            if tname == "get_weather":
                city = targs.get("city", targs.get("raw", "")).strip()
                data = await _fetch_weather(city)
                if data.get("cod") != 200:
                    resolved = call_llm(
                        "If the location given is a country name, return its capital city. "
                        "If it is already a city name, return it unchanged. "
                        "Reply with ONLY the city name.",
                        city, 0.0
                    ).strip().strip('"\'')
                    if resolved and resolved.lower() != city.lower():
                        print(f"       🌍 '{city}' → '{resolved}'", flush=True)
                        data = await _fetch_weather(resolved)
                results.append(format_weather(data))
                print(f"       🌤️ Weather for '{city}'", flush=True)
            else:
                results.append(f"Unknown tool: {tname}")
        final = get_llm(0.1).invoke([
            SystemMessage(content=(
                "You are a friendly weather assistant. "
                "Given weather data, write a clear natural weather report. "
                "Include temperature, conditions, humidity, wind. No [Tag] prefix."
            )),
            HumanMessage(content=f"User asked: {query}"),
            HumanMessage(content=f"Weather data:\n{chr(10).join(results)}\n\n"
                                  "Friendly weather report. No prefix tag."),
        ])
        return f"[Weather Agent]\n{strip_agent_prefix(final.content)}"

# ── STEP 13: DB Agent (cleaned, no _retry) ─────────────────────────────────────
class DBAgentExecutor(BaseAgentExecutor):
    _schema_cache: Optional[dict] = None

    @classmethod
    async def _load_schema(cls) -> dict:
        if cls._schema_cache is not None:
            return cls._schema_cache
        try:
            async with httpx.AsyncClient(timeout=5) as client:
                cls._schema_cache = (await client.get(f"{CFG.DB_API_URL}/schema")).json()
            print("       📋 [DB Agent] Schema loaded", flush=True)
        except Exception as e:
            print(f"       ⚠️ [DB Agent] Schema fetch failed: {e}", flush=True)
            cls._schema_cache = {}
        return cls._schema_cache

    @classmethod
    def _build_system_prompt(cls, schema: dict) -> str:
        tables = schema.get("tables", {})
        join_hint = schema.get("join_hint", "")

        def section(title: str, lines: list[str]) -> str:
            body = "\n".join(lines) or "  (none)"
            return f"{'='*55}\n{title}\n{'='*55}\n{body}"

        table_lines = [f"  {n}({', '.join(i.get('columns', []))})" for n, i in tables.items()]
        sample_lines = []
        for n, i in tables.items():
            for row in i.get("sample", []):
                sample_lines.append(f"    {n}: " + " | ".join(f"{k}={v}" for k, v in row.items()))
        enum_lines = []
        for n, i in tables.items():
            for k, v in i.items():
                if k.startswith("valid_") and isinstance(v, list):
                    enum_lines.append(f"  {n}.{k.replace('valid_','')}: {', '.join(repr(x) for x in v)}")
        text_cols = [f"{n}.{c.split()[0]}" for n, i in tables.items() for c in i.get("columns", []) if len(c.split())>=2 and c.split()[1].upper()=="TEXT"]
        text_hint = f"For TEXT use LOWER({text_cols[0]}) LIKE '%keyword%'" if text_cols else "Use LIKE for text"

        parts = [
            section("SCHEMA (live from DB API)", [
                "TABLES:", *table_lines, "", "SAMPLES:", *sample_lines, "", "ENUMS:", *enum_lines
            ]),
            section("SQL RULES", [
                "1. Only SELECT — never INSERT/UPDATE/DELETE/DROP.",
                "2. No backslash-escaped quotes.",
                f"3. Text search: {text_hint}",
                f"4. JOIN rule: {join_hint}",
                "5. COUNT: SELECT COUNT(*) FROM <table> WHERE ...",
            ]),
            section("INSTRUCTIONS", [
                "Use execute_sql tool (POSTs to DB API automatically).",
                "After results, write clean plain-text answer.",
                "Start final answer with exactly '[DB Agent]'.",
                "Do NOT include raw SQL in final answer.",
            ]),
        ]
        return "\n".join(parts)

    @classmethod
    def _build_retry_prompt(cls, schema: dict) -> str:
        tables = schema.get("tables", {})
        sigs = "; ".join(f"{n}({', '.join(i.get('columns',[]))})" for n,i in tables.items())
        enums = " | ".join(f"{n}.{k.replace('valid_','')}: {', '.join(repr(x) for x in v)}"
                          for n,i in tables.items() for k,v in i.items() if k.startswith("valid_") and isinstance(v,list))
        return (f"DB Agent. Tables: {sigs}. JOIN: {schema.get('join_hint','')}. Enums: {enums}. "
                "Only SELECT. No backslash quotes. Start with '[DB Agent]'.")

    async def direct_execute(self, query: str) -> str:
        schema = await self._load_schema()
        system = self._build_system_prompt(schema)
        tools = [execute_sql]
        try:
            response = await llm_invoke_with_retry(
                system, query, tools, self._build_retry_prompt(schema), "DB Agent", 0.0
            )
        except Exception as e:
            return f"[DB Agent]\n⚠️ LLM error: {e}"
        return await self._handle(query, system, response, tools, schema)

    async def _handle(self, query: str, system: str, response, tools: list, schema: dict) -> str:
        tcs = extract_tool_calls(response) or (parse_text_tool_calls(response.content) if hasattr(response,"content") else [])
        for tc in tcs:
            if tc["name"] == "execute_sql":
                sql = tc["args"].get("sql_query", "")
                print(f"       → DB API: {sql[:80]}", flush=True)
                raw = await call_db_api(sql)
                print(f"       ← Result: {raw[:120]}", flush=True)
                try:
                    final = get_llm(0.0).invoke([
                        SystemMessage(content=system),
                        HumanMessage(content=query),
                        HumanMessage(content=f"Database returned:\n{raw}\n\nWrite clean readable answer. Start with '[DB Agent]'. No SQL blocks."),
                    ])
                    content = final.content.strip()
                except Exception:
                    content = raw
                if not content.startswith("[DB Agent]"):
                    content = f"[DB Agent]\n{content}"
                return content
        content = response.content.strip() if hasattr(response, "content") else ""
        return content if content.startswith("[DB Agent]") else f"[DB Agent]\n{content}"

print("✅ All AgentExecutors ready", flush=True)

# ── STEP 14: Port & Server Utilities ────────────────────────────────────────────
class ServerManager:
    @staticmethod
    def is_free(port: int) -> bool:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            return s.connect_ex(("127.0.0.1", port)) != 0

    @staticmethod
    def kill_ports(*ports):
        for port in ports:
            if ServerManager.is_free(port):
                print(f"   ✅ Port {port} free", flush=True); continue
            killed = False
            try:
                for conn in psutil.net_connections(kind="inet"):
                    if conn.laddr.port == port and conn.pid:
                        try:
                            psutil.Process(conn.pid).kill()
                            print(f"   🔪 Killed PID {conn.pid} on {port}", flush=True)
                            killed = True
                        except (psutil.NoSuchProcess, psutil.AccessDenied):
                            pass
            except Exception as e:
                print(f"   ⚠️ psutil error on {port}: {e}", flush=True)
            if not killed:
                try:
                    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True, timeout=2)
                except Exception:
                    print(f"   ⚠️ Could not free port {port}", flush=True)

    @staticmethod
    async def wait_for(port: int, timeout: float = 10.0):
        waited = 0.0
        while ServerManager.is_free(port):
            await asyncio.sleep(0.2); waited += 0.2
            if waited > timeout:
                print(f"   ⚠️ Port {port} not up after {timeout}s", flush=True); break

    @staticmethod
    def start(app, port: int):
        cfg = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
        threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()

# ── STEP 15: JSON-RPC sender ────────────────────────────────────────────────────
async def send_jsonrpc(url: str, query: str) -> str:
    payload = {
        "jsonrpc": "2.0", "id": f"req_{int(time.time()*1000)}",
        "method": "tasks/send",
        "params": {
            "id": f"task_{int(time.time()*1000)}",
            "message": {"role": "user", "parts": [{"text": {"text": query}}]},
        },
    }
    async with httpx.AsyncClient(timeout=60) as client:
        data = (await client.post(url, json=payload)).json()
    for extractor in [
        lambda d: d["result"]["artifacts"][0]["parts"][0]["text"]["text"],
        lambda d: d["result"]["status"]["message"]["parts"][0]["text"]["text"],
        lambda d: d["result"]["status"]["message"]["parts"][0]["text"],
        lambda d: str(d["result"]),
    ]:
        try:
            return extractor(data)
        except Exception:
            pass
    return str(data)

# ── STEP 16: Orchestrator ───────────────────────────────────────────────────────
class OrchestratorAgentExecutor(BaseAgentExecutor):
    SYSTEM = "You are the A2A Orchestrator Agent."
    SUB_AGENT_URLS: list = []

    def __init__(self):
        self._agents: dict = {}
        self._discovered = False

    async def _discover(self):
        print("\n  🔍 [Orchestrator] Discovering sub-agents...", flush=True)
        async with httpx.AsyncClient(timeout=10) as client:
            for url in self.SUB_AGENT_URLS:
                card_dict = name = None
                try:
                    resolver = A2ACardResolver(httpx_client=client, base_url=url)
                    card_proto = await resolver.get_agent_card()
                    card_dict = MessageToDict(card_proto, preserving_proto_field_name=True)
                    name = card_proto.name
                    skills = [s.name for s in card_proto.skills]
                except Exception:
                    try:
                        r = await client.get(f"{url.rstrip('/')}{AGENT_CARD_WELL_KNOWN_PATH}")
                        card_dict = r.json()
                        name = card_dict.get("name", url)
                        skills = [s["name"] for s in card_dict.get("skills", [])]
                    except Exception as e2:
                        print(f"     ❌ Failed: {url} — {e2}", flush=True); continue
                emoji = AGENT_EMOJI.get(name, "🤖")
                print(f"     {emoji}  {name}  |  Skills: {skills}", flush=True)
                self._agents[name] = {
                    "url": url, "description": card_dict.get("description", ""),
                    "skills": [{"name": s.get("name",""), "description": s.get("description",""),
                                "tags": s.get("tags",[]), "examples": s.get("examples",[])}
                               for s in card_dict.get("skills", [])],
                }
        self._discovered = True
        print(f"     📋 {len(self._agents)} sub-agents ready\n", flush=True)

    def _build_catalogue(self) -> str:
        lines = []
        for name, info in self._agents.items():
            port = CFG.AGENT_PORTS.get(name, "?")
            lines.append(f'\n### {AGENT_EMOJI.get(name,"🤖")}  {name}  (port {port})')
            lines.append(f'  Description : {info["description"]}')
            for sk in info["skills"]:
                lines.append(f'  Skill : {sk["name"]} — {sk["description"]}')
                lines.append(f'    Tags : {", ".join(sk["tags"]) or "—"}')
                lines.append(f'    Examples : {" | ".join(sk["examples"][:2]) or "—"}')
        return "\n".join(lines)

    def _build_routing_examples(self) -> str:
        return "\n".join(f'  "{ex}" → {name}' for name, info in self._agents.items() for sk in info["skills"] for ex in sk["examples"][:2])

    def _build_intent_index(self) -> str:
        return "\n".join(f'  {name} → tags: [{", ".join(sorted({t for sk in info["skills"] for t in sk["tags"]}))}]'
                        for name, info in self._agents.items())

    def _build_planner_prompt(self) -> str:
        return f"""You are the master Orchestrator. Analyze the user query and produce a MINIMAL ordered execution plan.

{'='*55}
AVAILABLE AGENTS
{'='*55}
{self._build_catalogue()}

{'='*55}
ROUTING EXAMPLES
{'='*55}
{self._build_routing_examples()}

{'='*55}
AGENT INTENT INDEX
{'='*55}
{self._build_intent_index()}

{'='*55}
PLANNING RULES
{'='*55}
1. AGENT SELECTION — Use EXACT agent names from AVAILABLE AGENTS. Match intent via tags/skills.
2. MINIMUM STEPS — Prefer 1 step. NEVER call the same agent twice in a row. A single agent that can combine or relate multiple data sources counts as ONE step. Only add steps if the query EXPLICITLY requires sequential processing ("then write", "based on that").
3. PRESERVE FULL QUERY — The "task" field MUST be the user's COMPLETE exact question. NEVER shorten, rephrase, drop details, quantities, weights, or units.
4. DATA CHAINING — Only when step N+1 genuinely needs step N's output: embed {{step_N}} in task of step N+1.
5. OUTPUT — ONLY a valid JSON array. No markdown fences. No extra text.
   Single: [{{"step":1,"agent":"<name>","task":"<full task>"}}]
   Multi:  [{{"step":1,"agent":"<A>","task":"<task1>"}},{{"step":2,"agent":"<B>","task":"<task2 using {{step_1}}>"}}]
"""

    def _plan(self, query: str) -> list[dict]:
        system = self._build_planner_prompt()
        agent_names = "\n".join(f'- "{n}"' for n in self._agents)
        raw = call_llm(
            system,
            f'Available agents:\n{agent_names}\n\n'
            f'User query: "{query}"\n\n'
            f'Reply with ONLY the JSON array.',
            0.0,
        ).strip()

        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        raw = raw.strip()

        try:
            steps = json.loads(raw)
            validated = []
            for s in steps:
                agent = s.get("agent", "").strip()
                matched = next(
                    (n for n in self._agents
                     if n.lower() in agent.lower() or agent.lower() in n.lower()),
                    None
                )
                validated.append({
                    "step": len(validated) + 1,
                    "agent": matched or list(self._agents.keys())[0],
                    "task": s.get("task", "").strip(),
                })

            # GENERIC GUARDRAIL — no hardcoded examples or agent names
            # For single-step plans, the planner's only job is routing.
            # It must NEVER rewrite the user's query, because that drops
            # weights, units, names, and constraints. Forward verbatim.
            if len(validated) == 1:
                validated[0]["task"] = query

            if validated:
                return validated

        except Exception as e:
            print(f"   ⚠️ Plan parsing failed ({e}), falling back", flush=True)

        return [{"step": 1, "agent": self._route_single(query), "task": query}]

    def _route_single(self, query: str) -> str:
        agent_list = "\n".join(f'- "{n}": {i["description"]}' for n, i in self._agents.items())
        chosen = call_llm(
            "You are a routing assistant. Reply with ONLY the exact agent name. No other text.",
            f"Agents:\n{agent_list}\n\nQuery: \"{query}\"\nReply with ONLY the exact agent name.",
            0.0,
        ).strip()
        return next(
            (n for n in self._agents if n.lower() in chosen.lower() or chosen.lower() in n.lower()),
            list(self._agents.keys())[0]
        )

    async def _execute_plan(self, query: str) -> str:
        if not self._discovered:
            await self._discover()
        if not self._agents:
            return "[Orchestrator] ❌ No sub-agents available."
        steps = self._plan(query)
        is_multi = len(steps) > 1
        mode = "MULTI-STEP" if is_multi else "SINGLE-STEP"
        print(f"\n  📋 [Orchestrator] {mode} Plan — {len(steps)} step(s):", flush=True)
        for s in steps:
            port = CFG.AGENT_PORTS.get(s["agent"], "?")
            print(f"     {s['step']}. {AGENT_EMOJI.get(s['agent'],'🤖')}  {s['agent']}  →  {s['task'][:60]}", flush=True)

        step_results: dict = {}
        step_details: list = []
        for s in steps:
            task = s["task"]
            for prev_num, prev_result in step_results.items():
                task = task.replace(f"{{step_{prev_num}}}", prev_result)
            port = CFG.AGENT_PORTS.get(s["agent"], "?")
            print(f"     ⏳ Calling {AGENT_EMOJI.get(s['agent'],'🤖')}  {s['agent']} (port {port})...", flush=True)
            result = await send_jsonrpc(self._agents[s["agent"]]["url"], task)
            step_results[s["step"]] = result
            print(f"     ✅ {s['agent']} responded.", flush=True)
            step_details.append({
                "step": s["step"], "agent": s["agent"], "task": task, "result": result,
                "url": self._agents[s["agent"]]["url"], "meta": {"port": port},
            })

        response = self._build_response(query, step_details, is_multi)
        entry = chat_history.add(query, step_details, response, "multi" if is_multi else "single")
        print(f"     💾 Saved to history #{entry['id']}", flush=True)
        return response

    def _build_response(self, original_query: str, step_details: list, is_multi: bool) -> str:
        W = 62
        def box(text: str, left_pad: int = 2) -> str:
            visible = len(text) - text.count('🎯') - text.count('🔗') - text.count('➡️')
            pad = W - left_pad - visible - 2
            return f"║ {' '*left_pad}{text}{' '*max(0,pad)}║"
        lines = [f"╔{'═'*W}╗"]
        if is_multi:
            lines.append(box(f"🎯 ORCHESTRATOR  →  Multi-Step Plan ({len(step_details)} steps)"))
            lines.append(f"╠{'═'*W}╣")
            for d in step_details:
                lines.append(box(f"Step {d['step']}: {AGENT_EMOJI.get(d['agent'],'🤖')}  {d['agent']}  (port {d['meta']['port']})", left_pad=0))
        else:
            d = step_details[0]
            lines.append(box("🎯 ORCHESTRATOR  →  Routing Report"))
            lines.append(f"╠{'═'*W}╣")
            lines.append(box(f"Agent : {d['agent']}", left_pad=2))
            lines.append(box(f"Port  : {d['meta']['port']}", left_pad=2))
            lines.append(box(f"URL   : {d['url']}", left_pad=2))
        lines.append(f"╚{'═'*W}╝")
        for d in step_details:
            lines.extend(["", f"{AGENT_EMOJI.get(d['agent'],'🤖')}  {'Step '+str(d['step'])+' — ' if is_multi else ''}{d['agent']}:", "─"*W, d["result"], "─"*W])
        return "\n".join(lines)

    async def direct_execute(self, query: str) -> str:
        return await self._execute_plan(query)

print("✅ OrchestratorAgentExecutor ready!", flush=True)

# ── STEP 17: A2A Server Builder ─────────────────────────────────────────────────
def _extract_query(message_data: dict) -> str:
    for p in message_data.get("parts", []):
        if isinstance(p.get("text"), dict):
            t = p["text"].get("text", "")
            if t: return t
        if isinstance(p.get("text"), str) and p["text"]:
            return p["text"]
        if p.get("type") == "text" and p.get("text"):
            return p["text"]
    return ""

def build_agent_app(name, description, base_url, skills_config, executor: AgentExecutor):
    agent_card = ParseDict({
        "name": name, "description": description, "version": "1.0.0",
        "capabilities": {"streaming": False},
        "supported_interfaces": [{"url": base_url, "protocolBinding": "JSONRPC"}],
        "skills": [{"id": s["id"], "name": s["name"], "description": s["description"],
                    "tags": s.get("tags", []), "examples": s.get("examples", [])} for s in skills_config],
    }, AgentCard())
    task_store = InMemoryTaskStore()
    handler = DefaultRequestHandlerV2(
        agent_executor=executor, task_store=task_store, agent_card=agent_card,
        queue_manager=InMemoryQueueManager(),
        request_context_builder=SimpleRequestContextBuilder(task_store=task_store),
    )
    async def serve_card(_request):
        return JSONResponse(MessageToDict(agent_card, preserving_proto_field_name=True))
    async def handle_rpc(request: Request):
        body = await request.json()
        method, params = body.get("method", ""), body.get("params", {})
        query = _extract_query(params.get("message", {}))
        task_id = params.get("id", f"task_{int(time.time()*1000)}")
        if method in ("tasks/send", "message/send") and query:
            try:
                send_req = ParseDict(params, SendMessageRequest())
                result_obj = await handler.on_message_send(send_req, None)
                return JSONResponse(MessageToDict(result_obj, preserving_proto_field_name=True))
            except Exception:
                pass
            try:
                result_text = await executor.direct_execute(query)
            except Exception as err:
                result_text = f"[Error] {err}"
            return JSONResponse({
                "jsonrpc": "2.0", "id": body.get("id"),
                "result": {
                    "id": task_id, "status": {"state": "completed", "message": {"role": "agent", "parts": [{"text": {"text": result_text}}]}},
                    "artifacts": [{"parts": [{"text": {"text": result_text}}], "index": 0, "lastChunk": True}],
                },
            })
        return JSONResponse({"jsonrpc": "2.0", "id": body.get("id"), "error": {"code": -32601, "message": f"Method '{method}' not found"}})
    app = Starlette(debug=False, routes=[
        Route(AGENT_CARD_WELL_KNOWN_PATH, serve_card, methods=["GET"]),
        Route("/", handle_rpc, methods=["POST"]),
    ])
    app._agent_card = agent_card
    return app

print("✅ A2A server builder ready!", flush=True)

async def query_orchestrator_agent(query: str) -> str:
    return await send_jsonrpc(f"http://127.0.0.1:{CFG.ORCHESTRATOR_PORT}", query)

print("✅ Demo client ready!", flush=True)

# ── STEP 18: Main ────────────────────────────────────────────────────────────────
async def main():
    print("\n" + "="*65 + "\n   A2A MULTI-AGENT — Optimized & Fixed (Generic)\n" + "="*65, flush=True)

    ports = [CFG.DB_API_PORT, CFG.ORCHESTRATOR_PORT] + list(CFG.AGENT_PORTS.values())
    print("\n🔧 Freeing ports...", flush=True)
    ServerManager.kill_ports(*ports)
    await asyncio.sleep(1)

    # DB API
    print("⚡ Starting DB API...", flush=True)
    ServerManager.start(db_api_app, CFG.DB_API_PORT)
    await ServerManager.wait_for(CFG.DB_API_PORT)
    async with httpx.AsyncClient(timeout=5) as client:
        try:
            h = await client.get(f"{CFG.DB_API_URL}/health")
            s = await client.get(f"{CFG.DB_API_URL}/schema")
            print(f"   💚 Health: {h.json()}", flush=True)
            print(f"   📋 Tables: {list(s.json().get('tables',{}))}", flush=True)
        except Exception as e:
            print(f"   ⚠️ DB API check failed: {e}", flush=True)

    # Agent definitions — GENERIC FIXES APPLIED:
    # - Tool Agent: broader tags for "stored data" matching
    # - DB Agent: added "join" skill to signal multi-table capability
    agents_cfg = [
        ("Research Agent", "Answers factual questions. Use for: facts, explanations, definitions.",
         9001, [{"id":"qa","name":"Q&A","description":"Answer factual questions","tags":["research","facts"],"examples":["What is the capital of France?","When did WW2 end?"]},
                {"id":"explain","name":"Explain","description":"Explain complex topics","tags":["explanation"],"examples":["Explain quantum computing"]}],
         ResearchAgentExecutor()),
        ("Math Agent", "Solves arithmetic, algebra, statistics, calculus.",
         9002, [{"id":"arithmetic","name":"Arithmetic","description":"Basic math","tags":["math"],"examples":["Sum of 10,20,30"]},
                {"id":"stats","name":"Statistics","description":"Mean, median, etc.","tags":["math","stats"],"examples":["Average of 5,10,15"]},
                {"id":"sequences","name":"Sequences","description":"Fibonacci, factorial","tags":["math"],"examples":["Fibonacci of 10"]}],
         MathAgentExecutor()),
        ("Creative Writer Agent", "Writes poems, stories, emails, LinkedIn posts. ONLY route if user asks to write/create content.",
         9003, [{"id":"poem","name":"Poetry","description":"Write poems","tags":["creative","poem"],"examples":["Write a poem about AI"]},
                {"id":"story","name":"Story Writing","description":"Short stories","tags":["creative","story"],"examples":["Write a story about robots"]},
                {"id":"email","name":"Email Drafting","description":"Professional emails","tags":["creative","email"],"examples":["Write a professional email about delay"]},
                {"id":"linkedin","name":"LinkedIn Post","description":"Engaging posts","tags":["creative","linkedin"],"examples":["Write a LinkedIn post about AI"]}],
         CreativeWriterAgentExecutor()),
        ("Summary Agent", "Condenses long text. ONLY route if user asks: summarize, summary, key points.",
         9004, [{"id":"summarize","name":"Text Summarization","description":"Executive summaries","tags":["summary","condense"],"examples":["Create a one-page summary"]}],
         SummaryAgentExecutor()),
        # GENERIC FIX: Broader tags so "stored data" matches Tool Agent, not DB Agent
        ("Tool Agent", "Executes get_metal_price, save_data, get_data.",
         9005, [{"id":"tools","name":"Tool Execution","description":"Metal prices and in-memory storage","tags":["tools","prices","memory","storage","saved","retrieve"],"examples":["What is gold price?","Store: note","Show all stored data"]}],
         ToolAgentExecutor()),
        # GENERIC FIX: Added "join" skill to signal multi-table single-query capability
        ("DB Agent", "Queries company database via HTTP. Schema loaded live.",
         9006, [{"id":"sql","name":"Database Queries","description":"SQL SELECT via HTTP","tags":["sql","database","employees","projects"],"examples":["List all employees","Who is on the AI Agents team?"]},
                {"id":"join","name":"Multi-Table Queries","description":"Fetch related information across multiple tables in a single query","tags":["join","combine","single-query"],"examples":["Show active projects with employee names"]}],
         DBAgentExecutor()),
        ("Weather Agent", "Fetches LIVE weather for any city.",
         9007, [{"id":"weather","name":"Live Weather","description":"Current weather via OpenWeatherMap","tags":["weather","temperature","humidity","wind","city"],"examples":["What is the weather in London?","What is the weather in Italy?"]}],
         WeatherAgentExecutor()),
    ]

    apps = {}
    print("\n⚡ Starting all A2A agents...", flush=True)
    for name, desc, port, skills, exe in agents_cfg:
        apps[name] = build_agent_app(name, desc, f"http://127.0.0.1:{port}/", skills, exe)
        ServerManager.start(apps[name], port)

    orchestrator_exe = OrchestratorAgentExecutor()
    orchestrator_exe.SUB_AGENT_URLS = [f"http://127.0.0.1:{p}" for p in CFG.AGENT_PORTS.values()]
    orchestrator_app = build_agent_app(
        "Orchestrator Agent", "Master orchestrator. Plans and routes across specialist agents. Supports multi-step tasks.",
        f"http://127.0.0.1:{CFG.ORCHESTRATOR_PORT}/",
        [{"id":"plan","name":"Multi-Step Planning","description":"Breaks complex tasks into steps","tags":["orchestration","planning"],"examples":["Research moon landing then write a LinkedIn post"]},
         {"id":"route","name":"Intelligent Routing","description":"Routes to best agent","tags":["orchestration","routing"],"examples":["What is gold price?","List all employees"]}],
        orchestrator_exe,
    )
    ServerManager.start(orchestrator_app, CFG.ORCHESTRATOR_PORT)

    print("⏳ Waiting for agents...", flush=True)
    all_ports = {CFG.ORCHESTRATOR_PORT: ("🎯","Orchestrator Agent"), **{p:(AGENT_EMOJI.get(n,"🤖"),n) for n,p in CFG.AGENT_PORTS.items()}}
    for port, (emoji, label) in all_ports.items():
        await ServerManager.wait_for(port)
        print(f"   ✅ {emoji}  {label} (:{port}) listening", flush=True)

    print("\n✅ All services running:", flush=True)
    print(f"   🗃️  Port {CFG.DB_API_PORT} → DB API", flush=True)
    for port, (emoji, label) in all_ports.items():
        arrow = "  ← ALL queries go here" if port == CFG.ORCHESTRATOR_PORT else ""
        print(f"   {emoji}  Port {port} → {label}{arrow}", flush=True)

    # Demo queries
    queries = [
        ("What is the capital of Japan?", "single", "🔬 Research"),
        ("Calculate the sum of 10, 25, 30, 45", "single", "🧮 Math"),
        ("Write a poem about artificial intelligence", "single", "✍️ Writer"),
        ("What is gold metal price?", "single", "🛠️ Tool"),
        ("What is price of 10 kg silver?", "single", "🛠️ Tool (10 kg)"),
        ("What is price of 500 g gold?", "single", "🛠️ Tool (500 g)"),
        ("Store: Rajendra likes AI and Python", "single", "🛠️ Tool"),
        ("Show all stored data", "single", "🛠️ Tool"),
        ("List all employees", "single", "🗄️ DB"),
        ("Who is working on the Knowledge Graph Architecture project?", "single", "🗄️ DB"),
        ("Show active projects with employee names", "single", "🗄️ DB"),
        ("Who is on the AI Agents team?", "single", "🗄️ DB"),
        ("What is the weather in London?", "single", "🌤️ Weather"),
        ("What is the weather in Italy?", "single", "🌤️ Weather → Rome"),
        ("What is the weather in Dubai?", "single", "🌤️ Weather"),
        ("Find out when the first moon landing happened and write a short engaging LinkedIn post celebrating that anniversary.", "multi", "🔬 Research → ✍️ Writer"),
        ("What is the fibonacci of 8? Then write a short poem inspired by that number.", "multi", "🧮 Math → ✍️ Writer"),
        ("Research quantum computing and create a one-page summary.", "multi", "🔬 Research → 📄 Summary"),
        ("Get the current weather in Paris and write a short travel recommendation post based on it.", "multi", "🌤️ Weather → ✍️ Writer"),
    ]

    print("\n" + "="*65 + "\n   DEMO — Single-step and Multi-step queries\n" + "="*65, flush=True)
    for i, (query, qtype, expected) in enumerate(queries, 1):
        tag = "🔗 MULTI " if qtype == "multi" else "➡️  SINGLE"
        print(f"\n{'─'*65}\n  [{i:02d}/{len(queries)}] [{tag}]  Expected: {expected}\n  Query : {query}\n{'─'*65}", flush=True)
        print(f"\n{await query_orchestrator_agent(query)}", flush=True)

    print("\n" + "="*65 + "\n  ✅ DEMO COMPLETE!\n" + "="*65, flush=True)
    chat_history.show_stats()

    # Interactive
    print("\n" + "="*65 + "\n  💬 INTERACTIVE MODE\n  Type queries or: history | history N | history show N | history stats | history search X | history export | quit\n" + "="*65 + "\n", flush=True)
    while True:
        try:
            user_input = input("You → Orchestrator (9000): ").strip()
        except EOFError:
            break
        if not user_input:
            continue
        lower = user_input.lower()
        if lower in ("quit", "exit", "q"):
            print("👋 Bye!", flush=True); break
        elif lower == "history":
            chat_history.show_recent(5)
        elif lower.startswith("history "):
            parts = lower.split()
            if len(parts) == 2 and parts[1].isdigit():
                chat_history.show_recent(int(parts[1]))
            elif len(parts) == 3 and parts[1] == "show" and parts[2].isdigit():
                chat_history.show_entry(int(parts[2]))
            elif len(parts) == 2 and parts[1] == "stats":
                chat_history.show_stats()
            elif len(parts) == 2 and parts[1] == "export":
                chat_history.export_session()
            elif len(parts) >= 3 and parts[1] == "search":
                chat_history.search(" ".join(user_input.split()[2:]))
            else:
                print("  Unknown history command.", flush=True)
        else:
            print(f"\n{await query_orchestrator_agent(user_input)}\n", flush=True)

# ── RUN ───────────────────────────────────────────────────────────────────────────
await main()

📦 Installing dependencies...
✅ Dependencies installed!
✅ All imports OK!
✅ LangChain-Groq ready — Model: llama-3.1-8b-instant
📚 Loaded 19 record(s)
✅ ChatHistory ready!
🗄️ SQLite company.db seeded!
✅ DB API app built
✅ LangChain tools ready
✅ All AgentExecutors ready
✅ OrchestratorAgentExecutor ready!
✅ A2A server builder ready!
✅ Demo client ready!

   A2A MULTI-AGENT — Optimized & Fixed (Generic)

🔧 Freeing ports...
   ✅ Port 8998 free
   ✅ Port 9000 free
   ✅ Port 9001 free
   ✅ Port 9002 free
   ✅ Port 9003 free
   ✅ Port 9004 free
   ✅ Port 9005 free
   ✅ Port 9006 free
   ✅ Port 9007 free
⚡ Starting DB API...
   💚 Health: {'status': 'ok', 'db': 'company.db'}
   📋 Tables: ['employees', 'projects']

⚡ Starting all A2A agents...
⏳ Waiting for agents...
   ✅ 🎯  Orchestrator Agent (:9000) listening
   ✅ 🔬  Research Agent (:9001) listening
   ✅ 🧮  Math Agent (:9002) listening
   ✅ ✍️  Creative Writer Agent (:9003) listening
   ✅ 📄  Summary Agent (:9004) listening
   ✅ 🛠️  Tool Agent (: